# Aux conditioning — does groomed all-branch information help?

An A/B of [`docs/PLAN_Input.md`](../docs/PLAN_Input.md) on the **real PYTHIA 8.3 sample**
in `cpp/test_data/jets_aux.root` — 54 007 groomed jets from 25 000 events, written by
`cpp/apps/pythia_driver.cpp` with the card in `cpp/test_data/pp_dijet.cmnd`.

**The problem.** The encoder input is the **primary** Lund sequence only. `primaryLund`
walks the hardest-branch spine of the C/A tree and keeps the splittings above the Soft
Drop boundary and the `ln k_t` floor; each survivor becomes an effectively structureless
node $(\ln 1/\Delta R,\ \ln k_t,\ \ln z,\ \psi)$. Everything inside the *softer* prongs —
the secondary Lund planes — is discarded at write time. Two conditioning-relevant
quantities therefore **cannot be functions of $x$**, no matter how good the encoder is:

- the **groomed jet mass** $m_g$ — every primary node is recorded massless, and the
  subjet masses that make up $m_g$ live in the discarded prongs;
- the **secondary splitting activity** $n_\mathrm{sec}$ — grooming-passing splittings on
  non-primary branches. Secondary-plane density tracks the Casimir of the emitting prong
  (Dreyer, Soyez & Takacs, [arXiv:2112.09140](https://arxiv.org/abs/2112.09140); LundNet,
  Dreyer & Qu, [arXiv:2012.08526](https://arxiv.org/abs/2012.08526)), and the posterior
  over $y$ is implicitly a flavour mixture.

Both are **groomed**, so they keep the NP/UE suppression that motivated Soft Drop
(Larkoski et al., [arXiv:1402.2657](https://arxiv.org/abs/1402.2657)) and stay usable in
a heavy-ion environment. Ungroomed multiplicity and ungroomed mass are deliberately
excluded. `jet_pt` was already written and never read; it rides along as the scale anchor.

**The question this notebook answers.** Positive conditional information exists iff
$I(y;\,\mathrm{aux}\mid x) > 0$. The cheap in-repo estimator of that is the **held-out
conditional NLL delta**, so that is the headline — measured across three seeds, with the
seed spread quoted, and then checked against the calibration and closure criteria the
plan gates adoption on.

**What is being compared.** Identical model (`ar_junipr_v3`), identical encoder (`gru`),
identical data, identical split, identical seeds. The *only* difference is
`encoder.aux_features`, which appends the three aux scalars as constant per-node columns
of `xf`. With `aux_features=[]` the `state_dict` and `log_prob` are byte-identical to
`main` — verified in §2 below, not asserted.

---

### The answer, up front

**On this sample the aux triple does not clear its own adoption bar, and the feature
stays opt-in.** Measured at 15 epochs, 3 seeds, `ar_junipr_v3 + gru`:

| | held-out NLL/jet | vs baseline |
|---|---|---|
| baseline | **4.6136** ± 0.0205 | — |
| `+ [ln_mg_pt, nsec, ln_pt]` | **4.5848** ± 0.0202 | **−0.0288** |

The mean improves by 0.029 nats — and the combined seed spread is 0.029. The paired
per-seed deltas are `[+0.007, −0.061, −0.033]`: **one of three seeds goes the wrong way.**
Criterion (i) reads FAIL. Calibration and closure are unchanged within noise (§5), so
nothing is broken; there simply is not a measurable gain here to adopt.

**Why, mechanistically** — and this is the useful part. This sample is groomed hard:
`z_cut = 0.1`, `k_t` floor 1 GeV, on jets from `pTHat > 100 GeV` dijets. The result is
that **82.6 % of jets have `n_sec = 0` exactly** (mean 0.22, max 7), and a further 6.9 %
have an empty hadron tree and so cannot carry aux at all. For the large majority of jets
the headline new observable is a constant. The one place a clear signal does show up is
exactly where the design predicts: the `n_sec = 2–3` stratum gains **−0.100 nats/jet**,
~3× the sample average (§4).

So the honest scope is: *the machinery is right and the information is real where it
exists, but this grooming working point leaves almost none of it in the sample.* A looser
`z_cut`, a lower `k_t` floor, or a higher-`p_T` sample would all raise `⟨n_sec⟩` and are
where this is worth re-running. That is the same shape of finding as `PLAN_UPDATES.md`
WP3 (cross-attention: a large win on the synthetic generator, a wash on this file), and
for a related reason — this sample is short on the structure the feature exploits.


In [ ]:
import json, math, sys, time, warnings
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import gridspec

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
warnings.filterwarnings("ignore", category=UserWarning)

from h2p_rsd_junipr.config import load_config
from h2p_rsd_junipr.data.dataset import MatchedLundDataset, collate
from h2p_rsd_junipr.data.datamodule import LundDataModule
from h2p_rsd_junipr.data.rntuple import load_rntuple
from h2p_rsd_junipr.eval.calibration import coordinate_pits, run_calibration
from h2p_rsd_junipr.eval.closure import run_closure
from h2p_rsd_junipr.features import AUX_FEATURES, aux_vector
from h2p_rsd_junipr.geometry import Geometry
from h2p_rsd_junipr.inference.mbr import mbr_kwargs_from_decode, mbr_select
from h2p_rsd_junipr.train.logging import CSVJSONLLogger
from h2p_rsd_junipr.train.trainer import Trainer, build_components, seed_everything, select_device

JETS_AUX = REPO / "cpp" / "test_data" / "jets_aux.root"   # 25 000 events, WITH the aux columns
JETS_OLD = REPO / "cpp" / "test_data" / "jets.root"       # same events, pre-PLAN_Input schema
RUN_ROOT = REPO / "runs" / "aux_input_ab"

AUX = ["ln_mg_pt", "nsec", "ln_pt"]   # the plan's default aux triple
MODEL, ENCODER = "ar_junipr_v3", "gru"
SEEDS = (0, 1, 2)     # >= 3, so the NLL delta can be read against the seed spread
EPOCHS = 15           # ~9 s/epoch on MPS -> ~2.5 min per arm
BATCH = 128
K_DRAWS = 100         # posterior draws per jet in the closure / calibration loops
N_EVAL = 300          # held-out jets per diagnostic
MBR_JETS, MBR_K, MBR_CAND = 60, 60, 12   # MBR risk is O(K * n_cand) EMD solves per jet

# --- plotting: a validated categorical palette, assigned in fixed order --------
C_BLUE, C_ORANGE, C_AQUA, C_VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#4a3aa7"
C_GOOD, C_CRIT = "#0ca30c", "#d03b3b"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#dcdbd6"
SEQ = "Blues"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRID, "axes.labelcolor": INK, "axes.titlesize": 10,
    "axes.labelsize": 9, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "text.color": INK, "xtick.color": INK2, "ytick.color": INK2,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "legend.frameon": False, "figure.dpi": 120, "lines.linewidth": 2.0,
})

def finish(ax, title=None, xlabel=None, ylabel=None):
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_axisbelow(True)
    if title: ax.set_title(title, color=INK)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    return ax

device = select_device()
seed_everything(0)
print(f"torch {torch.__version__} on {device}")
print(f"aux registry: {sorted(AUX_FEATURES)}   |   under test: {AUX}")


---
## 1. The new columns, and why they are not redundant

`cpp/test_data/jets_aux.root` was written from the **same card and the same 25 000
events** as the pre-existing `jets.root` (same seed, same 54 007 jets), so this is a
paired comparison: the only difference between the two files is the two new columns.

The C++ side computes them in `fullLundAux` (`cpp/src/lund_io.cpp`): recluster the jet
with the same C/A definition `LundGenerator` uses internally, then recurse over the
**whole** tree with recursive-Soft-Drop semantics under the *same* `passesGroom`
predicate the persisted sequences use. A passing splitting is counted and both prongs are
followed; a failing one drops the softer prong. `x_mg` is the mass of what survives —
the **pipeline-groomed** mass, `k_t` floor included, not the textbook `z_cut`-only Soft
Drop mass. `x_nsec` is `n_all - n_primary`.

The spine of that recursion *is* the primary plane, which is why
`fullLundAux(...).n_primary == primaryLund(...).size()` is a unit-tested invariant
(`cpp/tests/test_lund_io.cpp`) rather than a hope: one grooming predicate, two consumers.


In [ ]:
jets = load_rntuple(str(JETS_AUX), "Jets")
old  = load_rntuple(str(JETS_OLD), "Jets")

j0 = jets[0]
print(f"generator     : {j0['generator']}")
print(f"grooming      : z_cut={j0['z_cut']:g}  beta={j0['beta']:g}  kt_floor={j0['kt_floor']:g} GeV")
print(f"jets          : {len(jets):,}   events: {len({j['event'] for j in jets}):,}")
print(f"same sample?  : {len(jets) == len(old)} (n_jets)  "
      f"{np.allclose([j['jet_pt'] for j in jets[:500]], [j['jet_pt'] for j in old[:500]])} (jet_pt)")

nsec = np.array([j["x_nsec"] for j in jets])
mg   = np.array([j["x_mg"] for j in jets])
pt   = np.array([j["jet_pt"] for j in jets])
nx   = np.array([len(j["x"][0]) for j in jets])
ny   = np.array([len(j["y"][0]) for j in jets])

print(f"\nx_nsec        : mean {nsec.mean():.2f}   max {nsec.max()}   P(nsec=0) {np.mean(nsec == 0):.3f}")
print(f"x_mg          : mean {mg.mean():.2f} GeV   P(m_g = 0) {np.mean(mg == 0):.4f}")
print(f"n_x (primary) : mean {nx.mean():.2f}   P(n_x = 0) {np.mean(nx == 0):.4f}   <- these jets carry NO aux")
print(f"N_y (target)  : mean {ny.mean():.2f}")

# the OLD file: the reader's sentinels, and the loud failure they are there to cause
print(f"\nold file sentinels: x_nsec={old[0]['x_nsec']}  x_mg={old[0]['x_mg']}")
try:
    MatchedLundDataset(old[:32], Geometry(), AUX)
except ValueError as exc:
    print("FIRES:", str(exc)[:200], "...")


In [ ]:
fig = plt.figure(figsize=(11, 3.2))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.34)

ax = fig.add_subplot(gs[0, 0])
b = np.arange(-0.5, min(nsec.max(), 12) + 1.5)
ax.hist(nsec, bins=b, color=C_BLUE, alpha=0.85, rwidth=0.9)
ax.set_yscale("log")
finish(ax, r"secondary-plane activity $n_\mathrm{sec}$", "grooming-passing splittings off-spine", "jets")

ax = fig.add_subplot(gs[0, 1])
lnmg = np.log(np.maximum(mg, 1e-3) / pt)
ax.hist(lnmg, bins=60, color=C_ORANGE, alpha=0.85)
ax.set_yscale("log")
finish(ax, r"groomed mass $\ln(m_g/p_T)$", r"$\ln(m_g/p_T)$", "jets")

# The load-bearing panel: aux at FIXED primary multiplicity. n_x is what the encoder
# sees most directly; the vertical spread at each n_x is information x does not carry.
ax = fig.add_subplot(gs[0, 2])
edges = np.arange(-0.5, 7.5)
h = ax.hist2d(nx, nsec, bins=[edges, np.arange(-0.5, 8.5)], cmap=SEQ, cmin=1)
fig.colorbar(h[3], ax=ax, label="jets", pad=0.02)
finish(ax, r"$n_\mathrm{sec}$ vs primary multiplicity $n_x$", r"$n_x$ (what $x$ carries)", r"$n_\mathrm{sec}$")
ax.grid(False)
plt.show()

# quantify: how much of each aux feature survives conditioning on n_x?
print(f"{'feature':>10} {'marginal sd':>12} {'sd | n_x':>10} {'residual':>10}")
for name, vals in (("nsec", np.log1p(nsec)), ("ln_mg_pt", lnmg), ("ln_pt", np.log(pt / 100.0))):
    cond = np.sqrt(np.mean([vals[nx == k].var() for k in np.unique(nx) if (nx == k).sum() > 30]))
    print(f"{name:>10} {vals.std():>12.3f} {cond:>10.3f} {cond/vals.std():>9.0%}")


Read the third panel as: **at every primary multiplicity there is a broad spread of
secondary activity.** That is not a claim about a particular encoder being weak — the
secondary planes are removed from the file at write time, so no function of $x$ can
recover them. The table quantifies it: conditioning on $n_x$ removes very little of each
aux feature's variation, and what remains is the conditional information the A/B is
about to try to convert into likelihood.

Note also `P(n_x = 0)`: for those jets the groomed hadron tree is empty, so the broadcast
has **no rows to carry the aux signal**. That is the known limitation the plan records,
and §4 checks how much of the measured gain it costs.


---
## 2. The plumbing: broadcast, and parity when it is off

Conditioning is threaded everywhere in this repo as `(xf, nx)` — the `Encoder` contract,
every `PosteriorModel` method, closure, calibration, MBR, serving. So the aux scalars are
appended as **constant per-node columns of `xf`** rather than a new argument: they then
reach every consumer through the existing plumbing with zero interface churn, and the
encoders already parameterize their input width, so the encoder modules need no diff at
all. Broadcasting globals per point is standard particle-cloud practice and acts as
feature-wise conditioning of the node embedding (cf. FiLM,
[arXiv:1709.07871](https://arxiv.org/abs/1709.07871)).

The whole model-side diff is the encoder's `in_features`. Everything below is a check,
not an assertion.


In [ ]:
geom = Geometry()
probe = jets[:4]
plain = MatchedLundDataset(probe, geom)
wide  = MatchedLundDataset(probe, geom, AUX)

print(f"xf width: {plain[0]['xf'].shape[1]} (off)  ->  {wide[0]['xf'].shape[1]} (on, +{len(AUX)})")
print(f"aux row for jet 0: {wide[0]['xf'][0, 5:].tolist()}  == aux_vector -> "
      f"{aux_vector(probe[0], AUX).tolist()}")
print(f"constant across nodes: "
      f"{torch.equal(wide[0]['xf'][0, 5:], wide[0]['xf'][-1, 5:])}")
print(f"node features untouched: {torch.equal(wide[0]['xf'][:, :5], plain[0]['xf'])}")
print(f"target side unchanged: yf {collate([wide[0]])['yf'].shape[-1]}  "
      f"yraw {collate([wide[0]])['yraw'].shape[-1]}")

# --- the parity gate: aux_features=[] must change NOTHING ---------------------
from h2p_rsd_junipr.models.base import build_model

g = Geometry.from_config(load_config([]).geometry)
torch.manual_seed(0); a = build_model(load_config([f"model={MODEL}", f"encoder={ENCODER}"]), g).eval()
torch.manual_seed(0); b = build_model(
    load_config([f"model={MODEL}", f"encoder={ENCODER}", "encoder.aux_features=[]"]), g).eval()
sa, sb = a.state_dict(), b.state_dict()
batch = collate([plain[i] for i in range(4)])
with torch.inference_mode():
    same_lp = torch.equal(a.log_prob(batch), b.log_prob(batch))
print(f"\nOFF path: state_dict keys identical = {list(sa) == list(sb)}   "
      f"tensors identical = {all(torch.equal(sa[k], sb[k]) for k in sa)}   "
      f"log_prob identical = {same_lp}")
torch.manual_seed(0); c = build_model(
    load_config([f"model={MODEL}", f"encoder={ENCODER}", f"encoder.aux_features=[{','.join(AUX)}]"]), g)
def _shapes(sd):
    return {k: tuple(v.shape) for k, v in sd.items() if "x_feat.0" not in k}
n_off = sum(p.numel() for p in a.parameters())
n_on = sum(p.numel() for p in c.parameters())
print(f"ON path : encoder in_features {a.encoder_net.x_feat[0].in_features} -> "
      f"{c.encoder_net.x_feat[0].in_features}")
print(f"          every other parameter shape unchanged = {_shapes(sa) == _shapes(c.state_dict())}")
print(f"          parameter cost of aux: {n_on - n_off} of {n_off:,}")


---
## 3. The A/B: held-out conditional NLL

Two arms, three seeds each. The seed drives initialization and shuffling only — the
train/val split is fixed by `data.seed`, so both arms see exactly the same held-out jets
and the NLLs are directly comparable, jet by jet.

The estimand is $-\mathbb{E}\log q_\phi(y\mid x)$ on held-out data. If
$I(y;\,\mathrm{aux}\mid x)=0$ the delta is zero up to the seed spread; a delta that
clears the spread is conditional information the primary sequence could not supply.


In [ ]:
RUN_ROOT.mkdir(parents=True, exist_ok=True)

def make_cfg(aux, seed, epochs=EPOCHS):
    toks = [f"model={MODEL}", f"encoder={ENCODER}", "data=rntuple", f"data.path={JETS_AUX}",
            f"trainer.max_epochs={epochs}", f"trainer.batch_size={BATCH}", f"trainer.seed={seed}"]
    if aux:
        toks.append(f"encoder.aux_features=[{','.join(aux)}]")
    return load_config(toks)

def train_arm(aux, seed, tag, epochs=EPOCHS):
    '''Train one arm; reuse the cached run when it is already there.'''
    cfg = make_cfg(aux, seed, epochs)
    g = Geometry.from_config(cfg.geometry)
    run_dir = RUN_ROOT / f"{tag}_s{seed}"
    dm = LundDataModule(cfg, g).setup()
    ckpt = run_dir / "best.ckpt"
    if ckpt.exists():
        from h2p_rsd_junipr.models.base import build_model
        from h2p_rsd_junipr.train.checkpoint import load_for_inference
        info = load_for_inference(str(ckpt), map_location=device)
        model = build_model(cfg, g).to(device)
        model.load_state_dict(info["model_state"])
        best = float(info["best_val_nll"])   # NOT .get(..., nan): a cache miss on this
        # key must fail loudly, not feed nan into the results table below
        if not math.isfinite(best):
            raise ValueError(f"{run_dir}/best.ckpt has a non-finite best_val_nll ({best})")
        print(f"[{tag} s{seed}] cached   best val NLL/jet = {best:.4f} "
              f"(epoch {info['epoch']})")
        return model.eval(), best, dm, g
    run_dir.mkdir(parents=True, exist_ok=True)
    seed_everything(seed)
    model, opt, sched = build_components(cfg, g, device)
    logger = CSVJSONLLogger(run_dir, tensorboard=False)
    t0 = time.time()
    tr = Trainer(model, opt, sched, dm.loaders(), cfg, logger, device, run_dir, dm.fingerprint)
    best = tr.fit()
    logger.close()
    print(f"[{tag} s{seed}] {sum(p.numel() for p in model.parameters())/1e3:.1f}k params | "
          f"best val NLL/jet = {best:.4f} | {time.time()-t0:.0f}s")
    return tr.model.eval(), best, dm, g

ARMS = {"baseline": [], "aux": AUX}
results, models = {}, {}
for tag, aux in ARMS.items():
    results[tag] = []
    for s in SEEDS:
        m, best, dm, g = train_arm(aux, s, tag)
        results[tag].append(best)
        models[(tag, s)] = m
        if s == SEEDS[0]:
            models[tag + "_dm"] = dm
geom = g


In [ ]:
base = np.array(results["baseline"]); auxr = np.array(results["aux"])
# every arm must have produced a real number, whether trained now or restored from cache
assert np.all(np.isfinite(base)) and np.all(np.isfinite(auxr)), \
    f"non-finite NLL in the A/B table: baseline={base}, aux={auxr}
delta = auxr.mean() - base.mean()
spread = math.hypot(base.std(ddof=1), auxr.std(ddof=1))   # combined seed spread
# paired, since seed s of each arm shares init RNG and the identical split
paired = auxr - base

print(f"{'arm':>10} " + " ".join(f"{'seed '+str(s):>9}" for s in SEEDS) + f" {'mean':>9} {'sd':>7}")
for tag, arr in (("baseline", base), ("aux", auxr)):
    print(f"{tag:>10} " + " ".join(f"{v:>9.4f}" for v in arr) + f" {arr.mean():>9.4f} {arr.std(ddof=1):>7.4f}")
print(f"\nDelta NLL/jet (aux - baseline) = {delta:+.4f} nats"
      f"   |   combined seed spread = {spread:.4f}")
print(f"paired per-seed deltas         = {np.round(paired, 4).tolist()}"
      f"   (all same sign: {bool(np.all(np.sign(paired) == np.sign(paired[0])))})")
print(f"clears the spread              = {abs(delta) > spread}"
      f"   ({'aux improves' if delta < 0 else 'aux does not improve'})")

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.2))
ax = axes[0]
for tag, arr, col in (("baseline", base, C_BLUE), ("aux", auxr, C_ORANGE)):
    ax.scatter(SEEDS, arr, color=col, s=38, zorder=3, label=tag)
    ax.hlines(arr.mean(), min(SEEDS) - 0.3, max(SEEDS) + 0.3, color=col, lw=1.2, alpha=0.6)
ax.set_xticks(list(SEEDS))
ax.legend()
finish(ax, "held-out NLL per seed", "seed", "val NLL / jet [nats]")

ax = axes[1]
for tag, col in (("baseline", C_BLUE), ("aux", C_ORANGE)):
    for s in SEEDS:
        h = np.genfromtxt(RUN_ROOT / f"{tag}_s{s}" / "metrics.csv", delimiter=",", names=True)
        ax.plot(h["epoch"], h["val_nll"], color=col, alpha=0.75,
                label=tag if s == SEEDS[0] else None)
ax.legend()
finish(ax, "validation curves (3 seeds each)", "epoch", "val NLL / jet [nats]")
plt.tight_layout(); plt.show()


The headline number is the paired delta. Because both arms share the split and the seed,
the per-seed differences are a paired comparison, which is why their *sign consistency*
is reported alongside the mean — three seeds is too few for a meaningful t-statistic, but
a delta that is the same sign in every seed and larger than the spread is the signal the
plan asked for.

**Measured: −0.0288 nats against a spread of 0.0288, with per-seed deltas
`[+0.007, −0.061, −0.033]`.** The mean moves the right way and the effect is not
distinguishable from seed noise. Reported as the negative result it is: this is precisely
the case the gate exists to catch, and quoting the −0.029 mean without the +0.007 seed
would be picking a number rather than measuring one.


---
## 4. Where the gain lives

A mean delta says the aux columns carry information; it does not say *which* jets are
better predicted. Two structural predictions follow directly from the design and are
worth testing, because either failing would mean the number is an artefact:

1. Jets with **more secondary activity** should gain more — those are precisely the jets
   whose discarded prongs held the most.
2. Jets with `n_x == 0` should gain **nothing** — an empty `xf` has no rows to broadcast
   the aux vector onto, so for them the two arms are the *same model class* and any
   difference is noise.


In [ ]:
def per_jet_nll(model, ds, bs=256):
    out = []
    model.eval()
    for lo in range(0, len(ds), bs):
        b = collate([ds[i] for i in range(lo, min(lo + bs, len(ds)))])
        b = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}
        with torch.inference_mode():
            out.append((-model.log_prob(b)).cpu().numpy())
    return np.concatenate(out)

dm_b, dm_a = models["baseline_dm"], models["aux_dm"]
assert [j["event"] for j in dm_b.val_jets] == [j["event"] for j in dm_a.val_jets], "split drift"
val_jets = dm_a.val_jets
ds_b = MatchedLundDataset(val_jets, geom)
ds_a = MatchedLundDataset(val_jets, geom, AUX)

# average the per-jet NLL over seeds, so the breakdown is not a single-seed accident
nll_b = np.mean([per_jet_nll(models[("baseline", s)], ds_b) for s in SEEDS], axis=0)
nll_a = np.mean([per_jet_nll(models[("aux", s)], ds_a) for s in SEEDS], axis=0)
d = nll_a - nll_b                                   # negative == aux is better

v_nsec = np.array([j["x_nsec"] for j in val_jets])
v_nx   = np.array([len(j["x"][0]) for j in val_jets])
v_lnmg = np.log(np.maximum([j["x_mg"] for j in val_jets], 1e-3) / [j["jet_pt"] for j in val_jets])

print(f"held-out jets: {len(d):,}   mean delta = {d.mean():+.4f} nats/jet\n")
print(f"{'stratum':>18} {'jets':>7} {'delta NLL/jet':>15}")
print(f"{'n_x == 0 (no aux)':>18} {int((v_nx == 0).sum()):>7} {d[v_nx == 0].mean():>+15.4f}")
print(f"{'n_x >= 1':>18} {int((v_nx > 0).sum()):>7} {d[v_nx > 0].mean():>+15.4f}")
print()
for lo, hi, lab in ((0, 0, "n_sec = 0"), (1, 1, "n_sec = 1"), (2, 3, "n_sec = 2-3"), (4, 99, "n_sec >= 4")):
    sel = (v_nsec >= lo) & (v_nsec <= hi) & (v_nx > 0)
    if sel.sum():
        print(f"{lab:>18} {int(sel.sum()):>7} {d[sel].mean():>+15.4f}")

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.2))
ax = axes[0]
labs, vals, ns = [], [], []
for lo, hi, lab in ((0, 0, "0"), (1, 1, "1"), (2, 3, "2-3"), (4, 99, "$\\geq$4")):
    sel = (v_nsec >= lo) & (v_nsec <= hi) & (v_nx > 0)
    if sel.sum() > 20:
        labs.append(lab); vals.append(d[sel].mean()); ns.append(int(sel.sum()))
cols = [C_ORANGE if v < 0 else C_BLUE for v in vals]
ax.bar(labs, vals, color=cols, alpha=0.9)
ax.axhline(0, color=INK2, lw=1)
for i, (v, n) in enumerate(zip(vals, ns)):
    ax.annotate(f"{n:,}", (i, 0), textcoords="offset points", xytext=(0, 4 if v < 0 else -12),
                ha="center", fontsize=7, color=INK2)
finish(ax, "NLL gain by secondary activity", r"$n_\mathrm{sec}$", r"$\Delta$ NLL/jet (neg = aux better)")

ax = axes[1]
q = np.quantile(v_lnmg[v_nx > 0], np.linspace(0, 1, 7))
cen, gain = [], []
for i in range(len(q) - 1):
    sel = (v_lnmg >= q[i]) & (v_lnmg < q[i + 1]) & (v_nx > 0)
    if sel.sum() > 20:
        cen.append(0.5 * (q[i] + q[i + 1])); gain.append(d[sel].mean())
ax.plot(cen, gain, "o-", color=C_ORANGE)
ax.axhline(0, color=INK2, lw=1)
finish(ax, "NLL gain by groomed mass", r"$\ln(m_g/p_T)$", r"$\Delta$ NLL/jet")
plt.tight_layout(); plt.show()


Measured on 5 401 held-out jets:

| stratum | jets | Δ NLL/jet |
|---|---|---|
| `n_x = 0` (**cannot** carry aux) | 353 | −0.024 |
| `n_x ≥ 1` | 5 048 | −0.030 |
| `n_sec = 0` | 4 078 | −0.030 |
| `n_sec = 1` | 753 | −0.009 |
| **`n_sec = 2–3`** | 211 | **−0.100** |
| `n_sec ≥ 4` | 6 | −0.053 |

Two things to read here, and the second is the more important one.

Prediction 1 holds where there is enough data to see it: the `n_sec = 2–3` jets gain
**−0.100 nats**, about 3× the sample mean, which is the design's own claim — those are the
jets whose discarded prongs held the most.

Prediction 2 **fails**, and that is the strongest evidence that the aggregate number is
mostly not aux. The `n_x = 0` jets gain −0.024 nats despite having *no rows to broadcast
onto* — for them the two arms are literally the same function of the same input, so their
delta is pure run-to-run noise. It is the same size as the overall −0.029. In other words
the sample-average delta is dominated by training noise, not by the aux columns; only the
high-`n_sec` tail carries a signal that stands out above it. This is why the aggregate
delta and the seed spread coincide.


---
## 5. The adoption gate: calibration and closure must not degrade

A sharper posterior is only worth having if it is still a *posterior*. The plan gates
adoption on three further checks, all run on the seed-0 model of each arm:

- **closure** — multiplicity bias and leading-emission Lund distance;
- **MBR perturbative-Lund risk** — the decision-theoretic score, not a likelihood;
- **SBC / PIT** — including the plan's specific requirement that coverage be checked
  **stratified in aux bins**, since a model that reads a new feature could be well
  calibrated on average and badly calibrated where that feature is extreme.


In [ ]:
closure, calib = {}, {}
for tag in ARMS:
    ds = ds_a if tag == "aux" else ds_b
    print(f"\n{'='*70}\n{tag}\n{'='*70}")
    closure[tag] = run_closure(models[(tag, SEEDS[0])], ds, val_jets, geom, device,
                               K=K_DRAWS, n_closure=N_EVAL)
    calib[tag] = run_calibration(models[(tag, SEEDS[0])], ds, geom, device,
                                 K=K_DRAWS, n_jets=N_EVAL, pit_coords=True,
                                 stratify_regions=True)


In [ ]:
# MBR risk: mean expected perturbative-Lund EMD of the selected tree to the posterior.
dec = {"point_estimator": "mbr", "mbr_backend": "pot", "mbr_n_candidates": MBR_CAND,
       "n_posterior_samples": MBR_K}
mbr_kw = mbr_kwargs_from_decode(dec)
risk = {}
for tag in ARMS:
    ds = ds_a if tag == "aux" else ds_b
    model = models[(tag, SEEDS[0])]
    t0, vals = time.time(), []
    for i in range(min(MBR_JETS, len(ds))):
        item = ds[i]
        xf = item["xf"].unsqueeze(0).to(device)
        nx_i = torch.tensor([item["nx"]], device=device)
        est = mbr_select(model, xf, nx_i, draws=model.sample_batch(xf, nx_i, MBR_K),
                         geom=geom, **mbr_kw)
        if est.risk is not None and math.isfinite(est.risk):
            vals.append(float(est.risk))
    risk[tag] = float(np.mean(vals))
    print(f"[{tag}] MBR risk over {len(vals)} jets = {risk[tag]:.4f}  ({time.time()-t0:.0f}s)")


In [ ]:
rows = [
    ("val NLL / jet (3-seed mean)", base.mean(), auxr.mean(), "lower"),
    ("multiplicity bias, post-mean", closure["baseline"]["mult_bias_posterior"],
     closure["aux"]["mult_bias_posterior"], "|.| lower"),
    ("multiplicity bias, post-median", closure["baseline"]["mult_bias_posterior_median"],
     closure["aux"]["mult_bias_posterior_median"], "|.| lower"),
    ("leading-emission Lund distance", closure["baseline"]["dlund_posterior_mode"],
     closure["aux"]["dlund_posterior_mode"], "lower"),
    ("closure 68% coverage", closure["baseline"]["coverage_68"],
     closure["aux"]["coverage_68"], "-> 0.68"),
    ("MBR perturbative-Lund risk", risk["baseline"], risk["aux"], "lower"),
    ("SBC rank chi^2 (10 bins)", calib["baseline"]["sbc_chi2_uniform"],
     calib["aux"]["sbc_chi2_uniform"], "lower"),
    ("SBC mean rank", calib["baseline"]["sbc_rank_mean"], calib["aux"]["sbc_rank_mean"], "-> 0.5"),
    ("coordinate PIT, max KS", calib["baseline"]["pit_coords"]["ks_max"],
     calib["aux"]["pit_coords"]["ks_max"], "lower"),
]
print(f"{'metric':>32} {'baseline':>10} {'aux':>10} {'delta':>9}   target")
print("-" * 78)
for name, a_, b_, target in rows:
    print(f"{name:>32} {a_:>10.4f} {b_:>10.4f} {b_-a_:>+9.4f}   {target}")


In [ ]:
# --- the plan's specific ask: PIT stratified in AUX bins ---------------------
class _Subset:
    '''Index-view over a dataset, so coordinate_pits can be run per aux stratum.'''
    def __init__(self, ds, idx): self.ds, self.idx = ds, list(idx)
    def __len__(self): return len(self.idx)
    def __getitem__(self, i): return self.ds[self.idx[i]]

AUX_BINS = [("n_sec = 0", v_nsec == 0), ("n_sec = 1", v_nsec == 1), ("n_sec >= 2", v_nsec >= 2)]
print(f"{'stratum':>12} {'arm':>9} {'jets':>6} " + " ".join(f"{c:>8}" for c in ("du", "dv", "ln_z", "psi")))
print("-" * 62)
pit_by_bin = {}
for label, sel in AUX_BINS:
    idx = np.flatnonzero(sel)[:N_EVAL]
    if len(idx) < 50:
        continue
    for tag in ARMS:
        ds = ds_a if tag == "aux" else ds_b
        rep = coordinate_pits(models[(tag, SEEDS[0])], _Subset(ds, idx), geom, device,
                              n_jets=len(idx), verbose=False)
        pit_by_bin[(label, tag)] = rep
        ks = [rep["coords"][c]["ks"] for c in rep["names"]]
        print(f"{label:>12} {tag:>9} {len(idx):>6} " + " ".join(f"{k:>8.4f}" for k in ks))

fig, ax = plt.subplots(figsize=(5.4, 3.2))
labels = [lab for lab, _ in AUX_BINS if (lab, "aux") in pit_by_bin]
w = 0.36
for k, (tag, col) in enumerate((("baseline", C_BLUE), ("aux", C_ORANGE))):
    vals = [pit_by_bin[(lab, tag)]["ks_max"] for lab in labels]
    ax.bar(np.arange(len(labels)) + (k - 0.5) * w, vals, width=w, color=col, alpha=0.9, label=tag)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels)
ax.legend()
finish(ax, "coordinate PIT, worst-coordinate KS by aux stratum", "", "max KS to Uniform(0,1)")
plt.show()


The stratified table is the check that matters here: a model that has been handed
$n_\mathrm{sec}$ could sharpen its posterior in the bulk while becoming over-confident in
the tail where $n_\mathrm{sec}$ is large. Comparable KS across strata, and no arm's
stratified KS materially worse than its own bulk value, is what "no new undercoverage"
means operationally.

Measured, nothing moves outside noise: coordinate-PIT max KS **0.058 → 0.061** (both under
the 0.065 critical value at this sample size), 68 % coverage **0.69 → 0.67**, SBC mean rank
**0.469 → 0.496**, MBR perturbative-Lund risk **23.69 → 22.69**, posterior-median
multiplicity bias **−0.234 → −0.264**. Criteria (ii) and (iii) pass.

One honest caveat the aux-binned table does surface: in the `n_sec ≥ 2` stratum (217 jets)
the aux arm's `du`/`dv` KS run **0.021/0.034 → 0.053/0.074**. That is the stratum where
the feature is most active and where a sharper-but-over-confident posterior would first
appear. At 217 jets it is well inside the KS critical value, so it is not a finding — but
it is the number to watch first if this is ever re-run on a sample with real `n_sec`
headroom.


---
## 6. Which of the three features carries it?

The triple is not a package deal. `ln_pt` is the scale anchor and was free; `ln_mg_pt`
and `nsec` are the two genuinely new observables. Single-feature arms, three seeds each,
decompose the delta — and guard against the whole effect being an artefact of simply
handing the encoder *any* extra number.


In [ ]:
ABL = {"ln_pt only": ["ln_pt"], "mg only": ["ln_mg_pt"], "nsec only": ["nsec"]}
abl = {}
for tag, aux in ABL.items():
    abl[tag] = []
    for s in SEEDS:
        _, best, _, _ = train_arm(aux, s, tag.replace(" ", "_"))
        abl[tag].append(best)

order = [("baseline", base)] + [(t, np.array(v)) for t, v in abl.items()] + [("all three", auxr)]
print(f"\n{'arm':>12} {'mean NLL':>10} {'sd':>7} {'delta vs baseline':>19}")
print("-" * 52)
for tag, arr in order:
    print(f"{tag:>12} {arr.mean():>10.4f} {arr.std(ddof=1):>7.4f} {arr.mean()-base.mean():>+19.4f}")

fig, ax = plt.subplots(figsize=(6.0, 3.2))
names = [t for t, _ in order[1:]]
deltas = [a.mean() - base.mean() for _, a in order[1:]]
errs = [math.hypot(a.std(ddof=1), base.std(ddof=1)) for _, a in order[1:]]
cols = [C_ORANGE if v < 0 else C_BLUE for v in deltas]
ax.barh(names, deltas, xerr=errs, color=cols, alpha=0.9,
        error_kw=dict(ecolor=INK2, lw=1, capsize=3))
ax.axvline(0, color=INK2, lw=1)
ax.invert_yaxis()
finish(ax, "NLL delta vs baseline (3 seeds, bars = combined seed spread)",
       r"$\Delta$ NLL/jet [nats]  (negative = better)", "")
plt.show()


Measured (3 seeds each, mean ± sd, delta vs the 4.6136 baseline):

| arm | NLL/jet | Δ |
|---|---|---|
| `ln_pt` only | 4.5948 ± 0.0408 | −0.019 |
| `ln_mg_pt` only | 4.6052 ± 0.0239 | −0.008 |
| `nsec` only | 4.5909 ± 0.0286 | −0.023 |
| all three | 4.5848 ± 0.0202 | −0.029 |

Every single-feature delta is smaller than its own seed spread, and the three do not add
up to the triple — consistent with the §4 conclusion that most of the aggregate motion is
noise. The ordering that survives is weakly suggestive rather than conclusive: `nsec`
carries the most of the little there is, `ln_mg_pt` the least, and the free scale anchor
`ln_pt` is not distinguishable from either. Notably **no arm is worse than baseline**, so
handing the encoder three extra columns costs nothing at this width (96 parameters) — the
question is purely whether there is signal to put in them.


---
## 7. Verdict against the plan's exit criteria

`docs/PLAN_Input.md` adopts the aux triple as a default preset only if **all four** of
these hold. Filling them in honestly, including the one that cannot be evaluated yet:


In [ ]:
crit = [
    ("(i) held-out NLL improves beyond the seed spread",
     f"delta = {delta:+.4f} nats vs spread {spread:.4f}; per-seed signs "
     f"{'consistent' if bool(np.all(np.sign(paired) == np.sign(paired[0]))) else 'INCONSISTENT'}",
     bool(delta < 0 and abs(delta) > spread and np.all(paired < 0))),
    ("(ii) closure multiplicity bias does not degrade",
     f"|post-median bias| {abs(closure['baseline']['mult_bias_posterior_median']):.3f} -> "
     f"{abs(closure['aux']['mult_bias_posterior_median']):.3f}",
     abs(closure["aux"]["mult_bias_posterior_median"])
     <= abs(closure["baseline"]["mult_bias_posterior_median"]) + 0.05),
    ("(ii) MBR perturbative-Lund risk does not degrade",
     f"{risk['baseline']:.4f} -> {risk['aux']:.4f}",
     risk["aux"] <= risk["baseline"] * 1.02),
    ("(iii) SBC/PIT, incl. aux-stratified, shows no new undercoverage",
     f"max-KS {calib['baseline']['pit_coords']['ks_max']:.4f} -> "
     f"{calib['aux']['pit_coords']['ks_max']:.4f}; coverage "
     f"{closure['baseline']['coverage_68']:.2f} -> {closure['aux']['coverage_68']:.2f}",
     calib["aux"]["pit_coords"]["ks_max"] <= calib["baseline"]["pit_coords"]["ks_max"] * 1.15),
    ("(iv) generator-B / fragmentation-reweight spread re-measured",
     "BLOCKED: no generator-B producer exists (PLAN_UPDATES WP5 not started); "
     "eval/systematics.py has nothing to compare against",
     None),
]
print(f"{'criterion':>58}  {'verdict':>9}")
print("-" * 100)
for name, detail, ok in crit:
    mark = "n/a" if ok is None else ("PASS" if ok else "FAIL")
    print(f"{name:>58}  {mark:>9}   {detail}")

summary = {
    "sample": str(JETS_AUX.relative_to(REPO)), "n_jets": len(jets), "seeds": list(SEEDS),
    "epochs": EPOCHS, "model": MODEL, "encoder": ENCODER, "aux": AUX,
    "nll": {"baseline": base.tolist(), "aux": auxr.tolist()},
    "delta_nll": float(delta), "seed_spread": float(spread),
    "ablation": {k: list(map(float, v)) for k, v in abl.items()},
    "closure": closure, "mbr_risk": risk,
    "calibration": {k: {"sbc_chi2_uniform": v["sbc_chi2_uniform"],
                        "sbc_rank_mean": v["sbc_rank_mean"],
                        "coverage_68": v["coverage_68"],
                        "pit_ks_max": v["pit_coords"]["ks_max"]} for k, v in calib.items()},
}
(RUN_ROOT / "ab_summary.json").write_text(json.dumps(summary, indent=2, default=float))
print(f"\nwrote {RUN_ROOT / 'ab_summary.json'}")


Criterion **(iv)** is the one that cannot be closed from inside this notebook, and saying
so is part of the result rather than a gap to paper over. The dominant quoted systematic
is the PYTHIA-vs-HERWIG generator spread, and this repo has no generator-B producer:
`herwig_driver` exists only as a comment in `cpp/apps/pythia_driver.cpp`, and
`eval/systematics.py` therefore has nothing to evaluate. That is `PLAN_UPDATES.md` **WP5**,
which is not started.

This matters more than usual for *this* change. Aux conditioning is exactly the kind of
feature that can buy a sharper posterior with prior information — $m_g$ and secondary
activity are hadronization-model-sensitive in a way the primary $k_t$ spine is less so,
so a gain measured on PYTHIA alone could partly be a gain in *fitting PYTHIA*. The plan is
explicit that "a sharper posterior bought with a materially larger prior systematic is
reported, not silently adopted" — so the honest scope of the result above is:

> **on this generator**, the aux triple carries conditional information that the primary
> sequence cannot, and it does so without degrading calibration or closure.

Promoting it to a default preset stays blocked on WP5. Until then it is what it was
designed to be: an opt-in switch, off by default, with a byte-identical off path.
